# Cortos desde su largo (sin GPU)

Hay episodios cortos que son un tramo exacto de un largo (`cortos.json`, `docs/cortos-y-largos.md`): 053–067 dentro del
**077**, 072–076 y 078–104 dentro del **105**, 106–129 dentro del **130**, 139–153 dentro del **154**. La fábrica genera
solo el largo; este cuaderno monta cada corto con los clips del largo (misma voz y mismas frases), su master y su
informe. **No usa GPU**: elige un entorno sin acelerador (*Entorno de ejecución → Cambiar tipo → CPU*).

Antes: el largo tiene que estar generado entero con la fábrica. Lo que ya está hecho no se repite. Si alguna
intervención de un corto no tiene pareja en el largo, la lista al final: se genera después con la fábrica normal sobre
ese corto (hará solo esas). La celda 3 revisa frase a frase lo montado. Ejecuta todo seguido.

In [ ]:
# 0. Instalación (para la verificación de la celda 3)
!pip install -q jiwer num2words

In [ ]:
# 1. Parámetros: los largos cuyos cortos se montan
LARGOS = ["2021/077-happiness"]

In [ ]:
# 2. Montar los cortos
import os, sys, subprocess, json
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/fabrica", "/content/repo/piloto", "/content/repo/criba"]
import comun, fabrica
comun.montar_drive()
comun.desconectar_si_falla()
CFG = fabrica.cargar_config()
RAIZ_EPISODIOS = Path(CFG["rutas"]["drive"]) / CFG["rutas"]["episodios"]
GRUPOS = json.loads(Path("/content/repo/cortos.json").read_text(encoding="utf-8"))["grupos"]

def cargar(ep, crear=True):
    base = Path("/content/repo/episodios") / ep
    tr = json.loads((base / "transcript.es.json").read_text(encoding="utf-8"))
    titulo_en = json.loads((base / "metadata.en.json").read_text(encoding="utf-8"))["titulo"]
    return fabrica.plan_episodio(tr, CFG), fabrica.carpeta_episodio(RAIZ_EPISODIOS, ep, titulo_en, crear=crear), tr["titulo"]

PENDIENTES = {}
for largo in LARGOS:
    plan_l, carpeta_l, _ = cargar(largo, crear=False)
    faltan = [p["fichero"] for p in plan_l if not (carpeta_l / "tts" / p["fichero"]).exists()]
    if faltan:
        print(f"{largo}: el largo aún no está entero ({len(faltan)} clips sin hacer); lo salto")
        continue
    print(f"=== {largo} → {len(GRUPOS[largo])} cortos")
    cortos = []
    for ep in GRUPOS[largo]:
        plan, carpeta, titulo = cargar(ep)
        cortos.append((ep, plan, carpeta, titulo))
    for ep, hechos, total, sin in fabrica.montar_cortos(largo, plan_l, carpeta_l, cortos, CFG):
        if sin:
            PENDIENTES[ep] = sin
print("RESULTADO:", "OK, todos montados" if not PENDIENTES else f"faltan intervenciones sin pareja: {PENDIENTES}")

In [ ]:
# 3. Verificar, frase a frase, los cortos montados: clip de origen, frases = transcript, y lo que oyó Whisper al
#    generarlas frente al texto (WER ≤ 0,15) y sin cola sobrante. No usa GPU. Se puede relanzar sola.
DUDOSOS = {}
for largo in LARGOS:
    cortos = [(ep, *cargar(ep, crear=False)) for ep in GRUPOS[largo]]
    for ep, n, problemas in fabrica.verificar_cortos(largo, cortos, CFG):
        print(f"  {ep}: {n} frases · " + ("bien" if not problemas else f"{len(problemas)} a revisar"))
        for pr in problemas:
            print("      -", pr)
        if problemas:
            DUDOSOS[ep] = problemas
print("VERIFICACIÓN:", "todo bien" if not DUDOSOS else f"{len(DUDOSOS)} cortos con algo que revisar: {list(DUDOSOS)}")

In [ ]:
# 4. Subir a Drive y desconectar
comun.desconectar()